In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint
import numpy as np
import os
import cv2
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Load dataset (assuming it's already downloaded and preprocessed)
def load_data(image_dir, mask_dir, img_size=(256, 256)):
    images = []
    masks = []
    image_list = os.listdir(image_dir)
    
    for img_name in image_list:
        img = cv2.imread(os.path.join(image_dir, img_name), cv2.IMREAD_COLOR)
        img = cv2.resize(img, img_size)
        img = img / 255.0
        images.append(img)

        mask_name = img_name.replace('.jpg', '_mask.jpg')
        mask = cv2.imread(os.path.join(mask_dir, mask_name), cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, img_size)
        mask = mask / 255.0
        masks.append(mask)

    images = np.array(images)
    masks = np.array(masks).reshape(-1, img_size[0], img_size[1], 1)
    return images, masks

# U-Net Architecture
def unet_model(input_size=(256, 256, 3)):
    inputs = layers.Input(input_size)
    
    # Contracting Path
    c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    c3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)

    c4 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(p3)
    c4 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(c4)
    p4 = layers.MaxPooling2D((2, 2))(c4)

    # Bottleneck
    c5 = layers.Conv2D(1024, (3, 3), activation='relu', padding='same')(p4)
    c5 = layers.Conv2D(1024, (3, 3), activation='relu', padding='same')(c5)

    # Expansive Path
    u6 = layers.Conv2DTranspose(512, (2, 2), strides=(2, 2), padding='same')(c5)
    u6 = layers.concatenate([u6, c4])
    c6 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(u6)
    c6 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(c6)

    u7 = layers.Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(c6)
    u7 = layers.concatenate([u7, c3])
    c7 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(u7)
    c7 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c7)

    u8 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c7)
    u8 = layers.concatenate([u8, c2])
    c8 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u8)
    c8 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c8)

    u9 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c8)
    u9 = layers.concatenate([u9, c1])
    c9 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u9)
    c9 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c9)

    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c9)

    model = models.Model(inputs=[inputs], outputs=[outputs])
    return model

# Compile the model
unet = unet_model()
unet.compile(optimizer=Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])

# Load data
image_dir = '/kaggle/input/brain-tumor-segmentation/images'  # Replace with the path to your images
mask_dir = '/kaggle/input/brain-tumor-segmentation/masks'    # Replace with the path to your masks
images, masks = load_data(image_dir, mask_dir)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(images, masks, test_size=0.2, random_state=42)

# Data augmentation
data_gen_args = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1, shear_range=0.1, zoom_range=0.1, horizontal_flip=True, fill_mode='nearest')
image_datagen = ImageDataGenerator(**data_gen_args)
mask_datagen = ImageDataGenerator(**data_gen_args)

# Fit the generator to the data
seed = 42
image_datagen.fit(X_train, augment=True, seed=seed)
mask_datagen.fit(y_train, augment=True, seed=seed)

train_image_generator = image_datagen.flow(X_train, batch_size=16, seed=seed)
train_mask_generator = mask_datagen.flow(y_train, batch_size=16, seed=seed)
val_image_generator = image_datagen.flow(X_val, batch_size=16, seed=seed)
val_mask_generator = mask_datagen.flow(y_val, batch_size=16, seed=seed)

def combined_generator(image_gen, mask_gen):
    while True:
        image_batch = next(image_gen)
        mask_batch = next(mask_gen)
        yield (image_batch, mask_batch)

# Combine generators for training and validation
train_generator = combined_generator(train_image_generator, train_mask_generator)
val_generator = combined_generator(val_image_generator, val_mask_generator)

# Train the model
history = unet.fit(train_generator, validation_data=val_generator, epochs=5, 
                   steps_per_epoch=len(X_train)//16, validation_steps=len(X_val)//16)

# Plot accuracy and loss
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()

plt.show()


In [ ]:
def segment_single_image(model, image_path, img_size=(256, 256)):
    # Load and preprocess the image
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)
    img_resized = cv2.resize(img, img_size)
    img_normalized = img_resized / 255.0
    img_input = np.expand_dims(img_normalized, axis=0)  # Add batch dimension

    # Make predictions
    pred_mask = model.predict(img_input)
    pred_mask = (pred_mask.squeeze() > 0.5).astype(np.uint8)  # Binarize the mask

    # Display the original image and the predicted mask
    plt.figure(figsize=(10, 5))
    
    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(pred_mask, cmap='gray')
    plt.title('Predicted Mask')
    plt.axis('off')

    plt.show()

# Example usage
image_path = '/kaggle/input/brain-tumor-mri-dataset/Training/pituitary/Tr-piTr_0000.jpg'  # Replace with your image path
segment_single_image(unet, image_path)


In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf


# Function to segment all images in a directory and highlight tumor areas
def segment_and_highlight_tumors(model, folder_path, img_size=(256, 256), alpha=0.5):
    # Get list of all images in the folder
    image_list = [f for f in os.listdir(folder_path) if f.endswith(('.jpg', '.png', '.jpeg'))]

    for img_name in image_list:
        # Load and preprocess the image
        img_path = os.path.join(folder_path, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        img_resized = cv2.resize(img, img_size)
        img_normalized = img_resized / 255.0
        img_input = np.expand_dims(img_normalized, axis=0)  # Add batch dimension

        # Make predictions
        pred_mask = model.predict(img_input)
        pred_mask = (pred_mask.squeeze() > 0.5).astype(np.uint8)  # Binarize the mask

        # Create a colored overlay
        overlay = np.zeros_like(img_resized)
        overlay[..., 0] = 0  # Blue channel
        overlay[..., 1] = 0  # Green channel
        overlay[..., 2] = 255  # Red channel (to highlight)

        # Combine original image with overlay
        highlighted_img = cv2.addWeighted(img_resized, 1, overlay, alpha, 0)

        # Apply the predicted mask to the highlighted image
        highlighted_img[pred_mask == 0] = img_resized[pred_mask == 0]

        # Display the original image and the highlighted image
        plt.figure(figsize=(10, 5))
        
        plt.subplot(1, 2, 1)
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title('Original Image')
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(cv2.cvtColor(highlighted_img, cv2.COLOR_BGR2RGB))
        plt.title('Highlighted Tumor Areas')
        plt.axis('off')

        plt.show()

# Example usage
folder_path = '/kaggle/input/brain-tumor-mri-dataset/Testing/glioma'  # Replace with your folder path
segment_and_highlight_tumors(unet, folder_path)


In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf

# Ensure TensorFlow is set to use GPU
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    # Set memory growth for the GPU
    try:
        for gpu in physical_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Using GPU for inference.")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU available, using CPU.")

# Function to segment all images in a directory
def segment_images_in_folder(model, folder_path, img_size=(256, 256)):
    # Get list of all images in the folder
    image_list = [f for f in os.listdir(folder_path) if f.endswith(('.jpg', '.png', '.jpeg'))]

    for img_name in image_list:
        # Load and preprocess the image
        img_path = os.path.join(folder_path, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        img_resized = cv2.resize(img, img_size)
        img_normalized = img_resized / 255.0
        img_input = np.expand_dims(img_normalized, axis=0)  # Add batch dimension

        # Make predictions
        pred_mask = model.predict(img_input)
        pred_mask = (pred_mask.squeeze() > 0.5).astype(np.uint8)  # Binarize the mask

        # Display the original image and the predicted mask
        plt.figure(figsize=(10, 5))
        
        plt.subplot(1, 2, 1)
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title('Original Image')
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(pred_mask, cmap='gray')
        plt.title('Predicted Mask')
        plt.axis('off')

        plt.show()

# Example usage
folder_path = '/kaggle/input/brain-tumor-mri-dataset/Testing/glioma'  # Replace with your folder path
segment_images_in_folder(unet, folder_path)
